*updated 11 Aug 2025, Julian Mak (whatever with copyright, do what you want with this)

### As part of material for OCES 4303 "AI and Machine Learning in Marine Science" delivered at HKUST

For the latest version of the material, go to the public facing [GitHub](https://github.com/julianmak/OCES4303_ML_ocean) page.

---
# 8. Introduction to neural networks

***Neural Networks*** form a large part of modern day machine learning. Recall that the problem of regression or supervised learning is to find $\mathcal{N}$ in $\mathcal{N}(X) = Y$ given inputs and outputs $X$ and $Y$. [***Uuniversal approximation theorems***](https://en.wikipedia.org/wiki/Universal_approximation_theorem) essentially say that with enough complexity these neural networks can approximate functions $f$ with sensible properties arbitrarily well.

Neural Networks by themselves could probably be several courses by themselves, but we only have three sessions here. The first of these looks at anatomy of what actually goes into neural networks and the principles behind how these networks are trained, illustrating these with particularly simple network architectures. The remaining lectures can be regarded as theme and variations on this: the details differ, but the ideas remain largely the same.

> ## Key Objective(s)
> 1. Basic anatomy of a neural network as an approximation of an operator
> 2. Training of neural networks via back-propagation and gradient descent
> 3. Perceptrons and Multi-Layer Perceptrons as a basis of neural networks

For most of the things introduced here I am going to stick with `sklearn`, which has some basic functionalities to do simple neural networks. For the more advance cases we will need something more bespoke; for this course we will use `pytorch`, which is introduced in the next session.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

---
## a) Background concepts

Below is a schematic of a neural network $N$ I cooked up for illustrating the key parts of a neural network and what it does.

<img src="https://i.imgur.com/LFTTTa5.jpeg" width="600" alt='schematic neural network'>

The basic idea of a neural network is a thing that takes input $X$ and predicts a $\hat{Y}$. In my case this neural network has one single ***hidden layer*** (the components between the two purple lines) consisting of two nodes denoted $h_1$ and $h_2$. The $h_1$ and $h_2$ do elementary operations on the $X = (x_1, x_2)$ as
\begin{equation*}
    h_1 = f(w_1 x_1 + w_3 x_2 + b_1), \qquad h_2 = f(w_2 x_1 + w_4 x_2 + b_2).
\end{equation*}

We have:
* $w_i$ are the ***weights*** associated with the nodes that transform the inputs in some way
* $b_i$ are the ***biases*** that get added
* $f$ is called an ***activation function*** that eats some numbers and spits out another number

In the end $N$ eats some numbers and spits some stuff out: this would be the ***feed-forward*** (the terminology is presumbly drawn from *control theory* in the engineering field).

To see a feedforward in action, I'll do one of these calculations, and the below will be a code version of this. Suppose $X = [-1, 1]$, and my weight vector is $w = (1, 2, 3, 4)$, my biases are $b = (0, 1)$, and I don't have a non-trivial activation function, i.e. I have the identify function $f(x) = x$ for all values. Then at $h_1$, I have the following intermediate steps:

1. $x_1 w_1 + x_2 w_3 + b_1 = (-1)(1) + (1)(3) + 0 = -1 + 3 + 0 = 2$
2. $f(x_1 w_1 + x_2 w_3 + b_1) = f(2) = 2$

So $h_1(X) = 2$. Convince yourself that $h_2(X) = 3$, thus $N(X) = \hat{Y} = 2 + 3 = 5$. A coded up version of it looks like the below: I lumped $w$ and $b$ into an array variable called $\theta$ (`theta`).

In [ ]:
def simple_nn(X, theta, activation="unity"):
    # pull out relevant numbers
    x_1, x_2 = X
    w_1, w_2, w_3, w_4 = theta[:4]
    b_1, b_2 = theta[4:]

    # compute arguments (note this could be written as a matrix multiplication)
    h_1 = w_1 * x_1 + w_3 * x_2 + b_1
    h_2 = w_2 * x_1 + w_4 * x_2 + b_2

    # pass through activation function
    match activation:
        case "unity":
            h_1, h_2 = h_1, h_2
        case "tanh":
            h_1, h_2 = np.tanh(h_1), np.tanh(h_2)

    # return final output
    return h_1 + h_2

X = (-1, 1)
theta = (1,2,3,4,0,1)

print(f"output of simple_nn(X) is {simple_nn(X, theta):.6f}")
print(" ")

### Activation functions

While I didn't do it above, try and convince yourself that if I don't specify a non-trivial ***activation function*** then most things I wrote above can in fact be written as a matrix multiplication. That's not entirely surprising in hindsight, because then my nodes and each of the hidden layers are just doing sums and additions, i.e. linear operations, and linear operations can be represented by matrices. Each hidden layer $\ell$ is represented by a matrix $A_\ell$, but then
\begin{equation*}
    A_1(X) = a_1, \quad A_2(a_1) = a_2, \quad \ldots A_M(a_{M-1}) = \hat{Y}
\end{equation*}
is really just
\begin{equation*}
    A_M(\ldots A_3(A_2(A_1(X)))) = A(X) = \hat{Y},
\end{equation*}
so our task just boils down to finding $A$.

However, we don't want to just stick with linear operations, because that is somewhat limited. One way that neural networks add in nonlinearity is through ***activation functions*** $f$. Then we have instead
\begin{equation*}
    A_1(X) = f(a_1), \quad A_2(f(a_1)) = f(a_2), \quad \ldots A_M(f(a_{M-1})) = \hat{Y}
\end{equation*}
but the chain $A_M(\ldots A_3(A_2(A_1))) = A$ no longer holds, because we now have nonlinearity. The universal approximation theorems apply here also, so we can build a wider variety of operators from simple components.

Common activations are illustrated below. Some of these you have seen already in some sense: they are just flipped versions of the one-sided loss functions from when we were doing classification. Indeed, the code below I literally just copied and pasted from a previous lecture and modified/deleted a few lines...

In [ ]:
# sample activation functions

slope = 0.1

X = np.linspace(-4, 4, 101)

fig = plt.figure(figsize=(6, 3))
ax = plt.axes()
ax.plot(X, 1.0 / (1.0 + np.exp(-X)), label="sigmoid/logistic")
ax.plot(X, np.tanh(X), label="tanh")
ax.plot(X, np.maximum(X, 0), label="ReLU")
ax.plot(X, slope * np.minimum(X, 0) + np.maximum(X, 0), label=f"leaky ReLU (slope={slope})",
        zorder=-1)  # force it to be below other plots
ax.set_ylim((-1.5, 4))
ax.legend()
ax.set_xlabel(r"$X$")
ax.set_ylabel(r"$f(X)$")
ax.grid();

The idea of activations are that they are at least piecewise-differentiable (for reasons to be detailed later), and the control the value of the outcome somewhat. The examples given above are:

* ***Sigmoid*** generally refers to something that is $S$-shaped, and one example is the ***logistic function*** demonstrated above. This example is bounded between [0, 1]
* ***Hyperbolic-tangent*** (or tanh) is like the above. The standard form is bounded between [-1, 1].
* ***Rectified Linear Unit*** or (ReLU) is like the flipped version of the hinge loss from before, and the standard form is bounded below by zero.
* Leaky ReLU is like ReLU but has an extra part with a slope in the negative part.

The thing with sigmoid and ReLU is that if the inputs are (sufficiently) negative then your just get zero out. This is refereed to as a neuron not "firing" (because it spits out nothing), and partly the reason why these are called activation functions.

In the subroutine above I implemented the `tanh` activation function. The intermediate steps we have are
\begin{equation*}
    h_1(X) = \tanh(2), \qquad h_2(X) = \tanh(3),
\end{equation*}
so $\mathcal{N}(X) \approx 1.959...$, which is also what is spat out by a call of the subroutine.

In [ ]:
X = (-1, 1)
theta = (1,2,3,4,0,1)

# this one needs to be single quotes in not match the double quotes...
print(f"output of simple_nn(X) is now {simple_nn(X, theta, activation='tanh'):.6f}")
print(" ")

> <span style="color:red">Q.</span> Put in sigmoid and ReLU activation in and see what outputs you get there (you should do these by hand to check you implemented it correctly of course).

### Back-propagation

So now we can do the feed-forward, how do we "train" the neural network $N$? Well again we need a measure of what it means for $N$ to do "well": we need to define a loss function $J$, possible with penalisations added in accordingly. 

The name of the game is as before: we want to find the control parameters $\theta = (w, b)$ of the neural network $N(\theta)$ such that the loss $J$ is minimised. As before, we can leverage (stochastic) gradient-based methods in the following way. I am going to put another copy of the schematic here for convenience of reading.

#### 1. Feed-forward (i.e. going from left to right) and evaluate loss $J$

Guess the initial $\theta$, do a feed-forward for the training set $X$ to generate $\hat{Y}$, and compute the loss $J(Y, \hat{Y})$. In the schematic, this is us going all the way from left to right.

#### 2. Form the optimisation problem

So we want to solve the problem (abusing notation substantially here) $\partial J / \partial \theta = 0$. Lets suppose we start with the first control variable $w_1$; thus we want to evaluate
\begin{equation*}
    \frac{\partial J}{\partial w_1}.
\end{equation*}

#### 3. Apply chain rule (i.e. tracing back from right to left)

Starting from all the way on the right, we see where $w_1$ is used. This would be in $h_1$, so since $h_1 = h_1(w_1)$, by chain rule we have
\begin{equation*}
    \frac{\partial J}{\partial w_1} = \frac{\partial J}{\partial h_1}\frac{\partial h_1}{\partial w_1}.
\end{equation*}
But then we also know that $h_1 = f(w_1 x_1 + \ldots)$, so by chain rule and product rule we have
\begin{equation*}
    \frac{\partial h_1}{\partial w_1} = f'(w_1 x_1 + \ldots) + x_1 f(w_1 x_1 + \ldots).
\end{equation*}
We are now back at the beginning. All the terms above can in principle be evaluated, particularly if we choose an activation function $f$ where the derivative is reasonably simple (e.g. no activation would mean $f'=1$, ReLU will give $f'=1$ for $x>0$ and $0$ otherwise). Continue with all the other entries in $\theta$.

#### 4. Do the root finding method

The result is a whole load of algebraic equations and you want to find the zeroes of that. That tells you how to update your $\theta$; update that, and then repeat until you there is some convergence or you get bored.

Then you can in principle see how you might do this for arbitrary-sized networks. Neural networks are constructed in such a way that ***back-propagation*** as described above by the chain rule is particularly easy to do, and is partly related to their success: gradient-based methods can be used, which speeds up convergence, so more control variables can be allowed in principle, which we can allow for more complexity in the neural network that allows for better representation of operators.

> <span style="color:red">Q.</span> Make up a set of inputs/targets and define a loss of your choice; MSE would be a particularly easy one. I would probably just do $Y = X$ or $Y = X^2$ or something like that, but use a non-trivial activation function (because otherwise you can solve the problem in one go with matrix inversion, which we will basically do now). Try and modify the weights in the subroutine manually to see how you might do it to reduce the loss.

---
## b) (Multi-Layer) Perceptrons

***Perceptrons*** is one of the early designs of a neural network. A schematic of this is shown below, for a case with a single layer and one with multiple layers (a MLP).

<img src="https://i.imgur.com/e4R4nMM.jpeg" width="600" alt='mlps'>

An earlier criticism of the single layer perceptrons is that it can only really do binary classifications on data that is linearly separable, and we can more or less do that already with SVMs. The above comment doesn't apply to the multiple layer case; MLPs should really be called perceptrons either, because they can do so much more.

Here we are going to make use of the cats and dogs data to demonstrate a few things with perceptrons. Note that the single layer perceptron basically has no hidden layers.

### 1. Single layer case with no activation function

In this case there is no nonlinearity and it really is just matrix inversion. For operator $A$, images $X$ and labels $Y$ we have
\begin{equation}
    AX = Y \quad \Rightarrow \quad A = YX^\dagger.
\end{equation}
Here $X^\dagger$ is the ***pseudo-inverse***. The arrays are unlikely going to be square, so the system is over-determined and we need to consider the problem as one of optimisation. The standard pseudo-inverse considers the problem as one of $L^2$ optimisation without a penalisation, i.e. linear regression.

Going to load the cats and dogs data and then train up the model in the old fashioned way of matrix inversion. I'm just going to use all the data for demonstration purposes.

> NOTE: We should not expect fantastic skill for the following problems with the cats and dogs data, because what I will be doing are inherently difficult tasks to do.

In [ ]:
# don't read the headers

option = "remote"

if option == "local":
    print("loading data locally (assumes file has already been downloaded)")
    path = "cat.csv"
elif option == "remote":
    print("loading data remotely")
    path = "https://raw.githubusercontent.com/julianmak/OCES4303_ML_ocean/refs/heads/main/cat.csv"
else:
    raise ValueError("INVALID OPTION: use 'remote' or 'local'")

df_cats = pd.read_csv(path, header=None).T # make "features" the axis=-1
X_cats = df_cats.values

if option == "local":
    print("loading data locally (assumes file has already been downloaded)")
    path = "dog.csv"
elif option == "remote":
    print("loading data remotely")
    path = "https://raw.githubusercontent.com/julianmak/OCES4303_ML_ocean/refs/heads/main/dog.csv"
else:
    raise ValueError("INVALID OPTION: use 'remote' or 'local'")

df_dogs = pd.read_csv(path, header=None).T # make "features" the axis=-1
X_dogs = df_dogs.values

In [ ]:
# generate a list of 25 indices (generate full list, shuffle, select first 25, so no repeats)
ind = np.arange(80)
np.random.shuffle(ind)  # syntax for shuffle: not used like a function with input output...

# sample show (on-the-fly reshape data)
fig = plt.figure(figsize=(8, 6.5))
for i in range(20):
    ax = plt.subplot(4, 5, i+1)
    if i+1 < 10:
        ax.imshow(np.reshape(X_cats[ind[i], :], (64, 64)).T, cmap="gray")
    else:
        ax.imshow(np.reshape(X_dogs[ind[i], :], (64, 64)).T, cmap="gray")
    ax.set_title(f"#{ind[i]}")
    ax.set_xticks([]); ax.set_yticks([]);

In [ ]:
# shape of "cats" and "dogs" here is (pixels, index), so it is already flattened

n = 64  # take the first 80% (out of 80 entries) of the data just because

X_train = np.concatenate((X_cats[:n, :], X_dogs[:n, :]), axis=0)  # combine
Y_train = np.concatenate((np.ones(n), -1*np.ones(n)))         # label: cats = 1 and dogs = -1

X_test = np.concatenate((X_cats[n::, :], X_dogs[n::, :]), axis=0)
Y_test = np.concatenate((np.ones(80-n), -1*np.ones(80-n)))

# scale and redefine the data
scaler = StandardScaler()
scaler.fit(X_train)
X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

# obtain model by simple matrix inversion
A_pinv = Y_train @ np.linalg.pinv(X_train.T)

Having got the model (which is just a matrix here), we make make predictions. We multiply the test data set and see what number it would give us. It won't give us the label values exactly (`1` and `-1`), but it essentially say that if it is positive/negative it is `1` and `-1` respectively, i.e. we only care about the sign.

In [ ]:
# test model by doing matrix multiplication (right answer is [1 1 1 ... -1 -1 -1])
Y_pred = np.sign(A_pinv @ X_test.T)  # just need the sign

# plot out the predictions (circles should lie on top of crosses if completely correct)
fig = plt.figure(figsize=(10, 2))
ax = plt.axes()
ax.plot(Y_pred, 'bx', label="predictions")
ax.plot(Y_test, 'ro', fillstyle="none", label="truth")
ax.plot([15.5, 15.5], [-1.3, 1.3], 'k--', alpha=0.7)
ax.set_xlabel("index")
ax.legend()
ax.grid()

# if Y_pred = +-1 and Y_test = +-1 (i.e. correct predictions) then Y_pred * Y_test = 1
accuracy = np.sum(Y_pred * Y_test == 1) / len(Y_pred * Y_test)
ax.set_title(f"0-NN (pinv) accracy = {accuracy*100}%");

The accuracy is not great, but image recognition is quite a hard problem.

The thing that is of possible interest is what is the model that was returned by the data? We can probe this by actually plotting it out: we show below the (normalised) coeffs as a 1d array, then reshaped into the image, and also plot a random image in the dataset for comparison reasons.

In [ ]:
# plot out normalised coefficients and "power" in the image
fig = plt.figure(figsize=(6, 4))
ax = plt.subplot2grid((4, 6), (0, 0), colspan=6)
ax.plot(A_pinv / np.max(np.abs(A_pinv)))
ax.set_ylabel(f"coeff")
ax.set_xticks([])

ax = plt.subplot2grid((4, 6), (1, 0), rowspan=3, colspan=3)
cs = ax.imshow(np.reshape(A_pinv, (64, 64)).T, cmap="gray")
ax.set_xticks([]); ax.set_yticks([]);
ax.set_title("coeffs as image")

ind = np.random.randint(X_test.shape[0])
ax = plt.subplot2grid((4, 6), (1, 3), rowspan=3, colspan=3)
ax.imshow(np.reshape(X_test[ind, :], (64, 64)).T, cmap="gray")
ax.set_xticks([]); ax.set_yticks([]);
ax.set_title(f"#{ind}")

plt.tight_layout(pad=0.05);

The way to think about the model $A$ as the image is that you take that $A$, multiply the image's pixels element-wise, then sum up the numbers to get a single number. If that number is bigger than 0 then the model predicts a cat, otherwise it's a dog.

The $L^2$ regression here gave us a lot of non-zero coefficients, which is also shown in the reshaped image. Where the colours are particularly white/black is showing where the model things the pixels are important for classifying the image as cat or dog. In that sense the model has interpretability in that it thinks certain pixels are more important than others.

We then recall that if we do $L^1$ penalisation then we could promote sparsity in the coefficients for this case. We can piggyback on `sklearn.linear_model.LASSO` in this case to do the same thing. The below code demonstrates how this would be done.

> NOTE: We can do this only because the perceptron basically has no hidden layers or activation functions.
>
> Here I turned down my regularisation parameter `alpha` from the default of `1`.

In [ ]:
# inversion with LASSO
from sklearn.linear_model import Lasso

# some optimum in relation to the regulatisation parameter it seems
model = Lasso(alpha=0.1).fit(X_train, Y_train)
A_lasso = model.coef_

# test model by doing matrix multiplication (right answer is [1 1 1 ... -1 -1 -1])
Y_pred = np.sign(A_lasso @ X_test.T)  # just need the sign

# plot out the predictions (circles should lie on top of crosses if completely correct)
fig = plt.figure(figsize=(10, 2))
ax = plt.axes()
ax.plot(Y_pred, 'bx', label="predictions")
ax.plot(Y_test, 'ro', fillstyle="none", label="truth")
ax.plot([15.5, 15.5], [-1.3, 1.3], 'k--', alpha=0.7)
ax.set_xlabel("index")
ax.legend()
ax.grid()

# if Y_pred = +-1 and Y_test = +-1 (i.e. correct predictions) then Y_pred * Y_test = 1
accuracy = np.sum(Y_pred * Y_test == 1) / len(Y_pred * Y_test)
ax.set_title(f"0-NN (LASSO) accracy = {accuracy*100}%");

In [ ]:
# plot out normalised coefficients and "power" in the image
fig = plt.figure(figsize=(6, 4))
ax = plt.subplot2grid((4, 6), (0, 0), colspan=6)
ax.plot(A_lasso / np.max(np.abs(A_lasso)))
ax.set_ylabel(f"coeff")
ax.set_xticks([])

ax = plt.subplot2grid((4, 6), (1, 0), rowspan=3, colspan=3)
cs = ax.imshow(np.reshape(A_lasso, (64, 64)).T, cmap="gray")
ax.set_xticks([]); ax.set_yticks([]);
ax.set_title("coeffs as image")

# ind = np.random.randint(X_test.shape[0])
ax = plt.subplot2grid((4, 6), (1, 3), rowspan=3, colspan=3)
ax.imshow(np.reshape(X_test[ind, :], (64, 64)).T, cmap="gray")
ax.set_xticks([]); ax.set_yticks([]);
ax.set_title(f"#{ind}")

plt.tight_layout(pad=0.05);

As expected there are a lot of zero coefficients, and the skill is actually slightly higher. Thing to note here is that the coefficients of the model are particularly non-zero

* near the eyes
* forehead
* near the mouth

This is interesting because this is possibly in line with our expectations that these might be key features that are useful for cats and dogs classification. Approaches such as these may be of interest to guide our feature creation for such problems (e.g. instead of throwing the whole image in, we may want to train models to label "eyes" and "mouth" and use those as features for our eventual classifier).

> <span style="color:red">Q.</span> Consider changing the `alpha` parameter and see how skill and model changes.
>
> <span style="color:red">Q.</span> Could try other penalisations also (e.g. elastic net). 

### 2. Using `sklearn` functionalities: classification

That was for the case of no activation function and no hidden layers, so the problem is particularly simple. We can add more complexity in. Going to stick with what is in `sklearn` for now and use `sklearn.neural_network.MLPClassifer` for this.

> NOTE: `MLPClassifier` I think by default requires you to have at least 1 hidden layer; the above has no hidden layers. All the nodes in the hidden layers are connected by construction.

Below code demonstrates some syntax for this. Going to start with the particularly simple cases first by using the default except for the activation function.

> NOTE: The defaults of interest and some to be discussed here are
> * `hidden_layer_sizes = (100,)` i.e. 1 hidden layer, 100 nodes
> * `activation = "relu"`, but I am going to override it to `"identity"` for the below
> * `solver = "adam"`, which is the ***adam*** solver of Kligma & Ba, a variant of SGD
> * `alpha = 1e-4` is the $L^2$ regularisation parameter; larger is more regularisation
> * `batch_size = "auto"` is the ***batch size*** to be discussed next session (default is 200 or `n_samples`, whichever is smaller); we don't have so much data at the moment to really need to talk about it...
> * `learning_rate = "constant"` is the ***learning rate*** or stepsize for `sgd`
> * `early_stopping = False` for ***early stopping***, to be discussed later

In [ ]:
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(random_state=1234,
                   activation="identity",
                  )
nn.fit(X_train, Y_train)
Y_pred = nn.predict(X_test)

# plot out the predictions (circles should lie on top of crosses if completely correct)
def MLP_plot(nn, Y_pred, Y_test):
    fig = plt.figure(figsize=(10, 2))
    ax = plt.axes()
    ax.plot(Y_pred, 'bx', label="predictions")
    ax.plot(Y_test, 'ro', fillstyle="none", label="truth")
    ax.plot([15.5, 15.5], [-1.3, 1.3], 'k--', alpha=0.7)
    ax.set_xlabel("index")
    ax.legend()
    ax.grid()
    
    # if Y_pred = +-1 and Y_test = +-1 (i.e. correct predictions) then Y_pred * Y_test = 1
    accuracy = np.sum(Y_pred * Y_test == 1) / len(Y_pred)
    ax.set_title(f"MLP (size = {nn.hidden_layer_sizes}, activation = {nn.activation})" +
                 f", accuracy = {accuracy*100}%");

    return fig
fig = MLP_plot(nn, Y_pred, Y_test)

Below shows a case where I change the activation function to ReLU, which actually leads to a minor degradation of skill.

In [ ]:
nn = MLPClassifier(random_state=1234,
                   activation="relu",
                  )
nn.fit(X_train, Y_train)
Y_pred = nn.predict(X_test)

# plot out the predictions (circles should lie on top of crosses if completely correct)
fig = MLP_plot(nn, Y_pred, Y_test);

Below is a case where I change the MLP to have two hidden layers, first one with 200 nodes (default was 100) and the second with 50 nodes.

> NOTE: I haven't tried very hard but I can't seem to get an accuracy beyond about 75%, at least not robustly. Increases in the hidden size doesn't seem to do much, and the main skill seems to come from having that second layer. The `relu` option in this case also seems more fickle than if I were to just use `identity`.

In [ ]:
nn = MLPClassifier(random_state=1234,
                   hidden_layer_sizes=(200, 50),
                   activation="identity",
                  )
nn.fit(X_train, Y_train)
Y_pred = nn.predict(X_test)

# plot out the predictions (circles should lie on top of crosses if completely correct)
fig = MLP_plot(nn, Y_pred, Y_test);

There are various things that can be probed from the model training. One would be the ***loss curve***, which shows you the change in the value of the loss per iteration (or ***training epoch***). This is nested within the trained model as `nn.loss_curve_`.

In [ ]:
# plot the loss curve associated with last model trained

fig = plt.figure(figsize=(6, 3))
ax = plt.axes()
ax.plot(nn.loss_curve_)
ax.set_xlabel("epoch index")
ax.set_ylabel("loss")
ax.grid()
ax.set_title(f"MLP (size = {nn.hidden_layer_sizes}, activation = {nn.activation})")
print(f"early stopping condition = {nn.early_stopping}")
print(" ")

As we can see the loss function fluctuates around as training progresses, but eventually tapers off and reaches some convergence criterion (e.g. the loss being low enough, the loss not changing very much after some time).

Sometimes the default convergence criterion is not triggered but we still want to kill the training off. This is called ***early stopping***, but I haven't switched this on here.

> <span style="color:red">Q.</span> Have a look at what the model internally does when `early_stopping = True`. The manual or querying the model as `nn?` is a good place to start.

The weights and biases associated with the nodes can in principal be queried by `nn.coefs_[i]`, where `i` would be the weights going from one layer to another. If the model is sufficiently complex then these may or may not be that useful to look at; I suppose the ones with the large values might suggest a certain pixel/feature is more dominant.

### 3. Using `sklearn` functionalities: regression

Here are we going to continue with a hard problem, which is to predict one half of the face with the other. Just for convenience I am going to do it only for cats; you could try and do it for cats and dogs.

In [ ]:
from sklearn.neural_network import MLPRegressor

# subroutine to split top and bottom half of pixels: reshape, split, then flatten for sklearn
def split_top_bottom(data):
    n, width = data.shape[0], int(np.sqrt(data.shape[1]))
    data = np.reshape(data, (n, width, width))
    top_half, bottom_half = data[:, :, 0:width//2], data[:, :, width//2::]
    top_half = np.reshape(top_half, (n, width*width//2))
    bottom_half = np.reshape(bottom_half, (n, width*width//2))

    return top_half, bottom_half

# shape of "cats" here is (pixels, index), so it is already flattened
n = 64

# manually split up into train and test
X_train, Y_train = split_top_bottom(X_cats[:n, :])
X_test, Y_test = split_top_bottom(X_cats[n::, :])

# scale the data
scale_X, scale_Y = StandardScaler().fit(X_train), StandardScaler().fit(Y_train)
X_train, X_test = scale_X.transform(X_train), scale_X.transform(X_test)
Y_train, Y_test = scale_Y.transform(Y_train), scale_Y.transform(Y_test)

In [ ]:
# generate a list of 25 indices (generate full list, shuffle, select first 25, so no repeats)
ind = np.arange(n)
np.random.shuffle(ind)  # syntax for shuffle: not used like a function with input output...

# sample show (on-the-fly reshape data)
fig = plt.figure(figsize=(8, 2))
for i in range(5):
    ax = plt.subplot(2, 5, i+1)
    ax.imshow(np.reshape(X_train[ind[i], :], (64, 32)).T, cmap="gray")
    ax.set_title(f"#{ind[i]}")
    if i == 0:
        ax.set_ylabel("$X$")
    ax.set_xticks([]); ax.set_yticks([]);

    ax = plt.subplot(2, 5, i+1+5)
    ax.imshow(np.reshape(Y_train[ind[i], :], (64, 32)).T, cmap="gray")
    if i == 0:
        ax.set_ylabel("$Y$")
    ax.set_xticks([]); ax.set_yticks([]);

I am going to predict bottom half from the top half. Note this is probably a harder task than the other way round.

In [ ]:
nn = MLPRegressor(max_iter=1000)  # to shut the iteration not converged warning
nn.fit(X_train, Y_train)

ind_train = np.arange(n)
np.random.shuffle(ind_train)  # syntax for shuffle: not used like a function with input output...

ind_test = np.arange(80-n)
np.random.shuffle(ind_test)

fig, ax = plt.subplots(figsize=(10, 4), nrows=2, ncols=6)

for j in range(0, 3):
    ind = ind_train[j]
    ax[0][j].imshow(np.hstack((np.reshape(X_train[ind, :], (64, 32)), 
                               np.reshape(Y_train[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"train {j}")

    X_in = X_train[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(X_in, (64, 32)), 
                               np.reshape(Y_pred, (64, 32)))).T, 
                    cmap="gray")

for j in range(3, 6):
    ind = ind_test[j]
    ax[0][j].imshow(np.hstack((np.reshape(X_test[ind, :], (64, 32)), 
                               np.reshape(Y_test[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"test {j-3}")

    X_in = X_test[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(X_in, (64, 32)), 
                               np.reshape(Y_pred, (64, 32)))).T, 
                    cmap="gray")
for i in range(2):
    for j in range(6):
        ax[i][j].set_xticks([]); ax[i][j].set_yticks([]);
        if j == 0:
            ax[0][j].set_ylabel("original")
            ax[1][j].set_ylabel("MLP")

Predictions look a bit sad/cursed, but the key features are there at least. If you are lucky you might hit the cases where the top half of the image has parts of the eye there, and the prediction creates weird eyes...

Going to see what happens if we use bottom to predict the top instead.

In [ ]:
# manually split up into train and test
Y_train, X_train = split_top_bottom(X_cats[:n, :])
Y_test, X_test = split_top_bottom(X_cats[n::, :])

# scale the data
scale_X, scale_Y = StandardScaler().fit(X_train), StandardScaler().fit(Y_train)
X_train, X_test = scale_X.transform(X_train), scale_X.transform(X_test)
Y_train, Y_test = scale_Y.transform(Y_train), scale_Y.transform(Y_test)

nn = MLPRegressor(max_iter=1000)  # to shut the iteration not converged warning
nn.fit(X_train, Y_train)

# do plot but use the same randomly chosen indices as above
fig, ax = plt.subplots(figsize=(10, 4), nrows=2, ncols=6)

for j in range(0, 3):
    ind = ind_train[j]
    ax[0][j].imshow(np.hstack((np.reshape(Y_train[ind, :], (64, 32)), 
                               np.reshape(X_train[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"train {j}")

    X_in = X_train[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(Y_pred, (64, 32)),
                               np.reshape(X_in, (64, 32)))).T, 
                    cmap="gray")

for j in range(3, 6):
    ind = ind_test[j]
    ax[0][j].imshow(np.hstack((np.reshape(Y_test[ind, :], (64, 32)), 
                               np.reshape(X_test[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"test {j-3}")

    X_in = X_test[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(Y_pred, (64, 32)),
                               np.reshape(X_in, (64, 32)))).T, 
                    cmap="gray")
for i in range(2):
    for j in range(6):
        ax[i][j].set_xticks([]); ax[i][j].set_yticks([]);
        if j == 0:
            ax[0][j].set_ylabel("original")
            ax[1][j].set_ylabel("MLP")

Still looking a bit funny, but maybe slightly less cursed than before (because a cursed ear looks less bad than a cursed face...)

Just going to add a little bit of complexity for demonstration purposes.

In [ ]:
# MLP with more complexity
nn = MLPRegressor(hidden_layer_sizes=(100, 100, 100),   # don't have this too big...
                  activation="relu",                # I find identity seems ok actually
                  solver="adam",
                  max_iter=1000)
nn.fit(X_train, Y_train)

fig, ax = plt.subplots(figsize=(10, 4), nrows=2, ncols=6)

for j in range(0, 3):
    ind = ind_train[j]
    ax[0][j].imshow(np.hstack((np.reshape(Y_train[ind, :], (64, 32)), 
                               np.reshape(X_train[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"train {j}")

    X_in = X_train[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(Y_pred, (64, 32)),
                               np.reshape(X_in, (64, 32)))).T, 
                    cmap="gray")

for j in range(3, 6):
    ind = ind_test[j]
    ax[0][j].imshow(np.hstack((np.reshape(Y_test[ind, :], (64, 32)), 
                               np.reshape(X_test[ind, :], (64, 32)))).T, 
                    cmap="gray")
    ax[0][j].set_title(f"test {j-3}")

    X_in = X_test[ind, :].reshape(1,-1)
    Y_pred = nn.predict(X_in)
    ax[1][j].imshow(np.hstack((np.reshape(Y_pred, (64, 32)),
                               np.reshape(X_in, (64, 32)))).T, 
                    cmap="gray")
for i in range(2):
    for j in range(6):
        ax[i][j].set_xticks([]); ax[i][j].set_yticks([]);
        if j == 0:
            ax[0][j].set_ylabel("original")
            ax[1][j].set_ylabel("MLP")

> <span style="color:red">Q.</span> You can see the horizontal line in the images even in the "original"s, and that's almost certainly to do with the application of scaling. Tidy this up a bit by undoing the scaling accordingly. (Pipeline: train/split, define scaler, scale, train, predict, then you unscale the input/predicted data for plotting.)
>
> <span style="color:red">Q.</span> Be more quantitative than I was in defining what it means for the regressor to be good (or not).
>
> <span style="color:red">Q.</span> I haven't tried very hard in exploring the possible model parameters. Try varying those yourself and see how the predictions change qualitatively and/or quantitatively. In particular try to vary the activation function, the regularisation parameter and the solver. Do the usual cross-validation on these accordingly.
>
> Main constraint you should be careful about is the neural network size: MLPs by construction are all fully connected, so your parameter space increases very quickly with the number of nodes/layers.

----------------
# More involved exercises with this notebook

## 1) Code your own neural network by hand

Define appropriate inputs and outputs (e.g. $Y = X$, $Y = X^2$, $Y = \sin (X)$), use a non-trivial activation function and a loss function (MSE would be a good one). Link up the `simple_nn` subroutine with `scipy.optimize` to optimize for the parameters; the default L-BFGS solver will probably be fine.

If you've done that then you've successfully coded your own (simple) neural network by hand.

## 2) Penguins data

Try doing the classifer and regressors with the penguins data. I seriously doubt you need that much complexity in the networks for this though: the feature space dimension is low, and number of samples is not that high (in contrast to the cats + dogs data above, where the raw feature space dimension is large ($64^2$) but the sample size is probably not large enough to balance that out.

## 3) Turtle and penguins data

The Kaggle dataset of [penguins and turtles](https://www.kaggle.com/datasets/abbymorgan/penguins-vs-turtles) encountered previously might be better to do for neural network training; try doing stuff on that.

## 4) Time series data

Consider training regressors for doing time-series prediction, using the `elnino34_sst.data`, or from model data (e.g. Lotka-Volterra, Lorenz etc.)

If you want, try and do classifying problems, e.g. using a few data points before to predict whether there will be an upcoming El-Nino / La Nina episode. This requires you to provide labels to the data first, but you should know how to do that in principle already from the exercises in the previous sessions.

## 5) More data is always a good thing?

Redo the above cats and dogs classification problem but only using the top or bottom half of the image. I personally find I can get high accuracy with a smaller/simpler network than if I throw in all the data (I can get up to 80% accuracy).

Possibly a demonstration that working smarter at the data processing stage can potentially yield more skill than throwing the whole kitchen sink in through the model complexity (cf. a well-designed experiment likely beats the fancy statistical tests you can do on the resulting data).

## 6) Extended cats dataset

This may require a bit more computing power than you would have in a regular instance, and may be worth skipping for now but revisiting in the next session when we deal with Convolutional Neural Networks. The extended cats dataset have 2000 images in (instead of the basic one with 80), which may or may not be better for training a regressor on. Try and train a regressor accordingly and quantify the skill in a way of your choosing.

You should note there are images within that dataset that may want to be excluded accordingly, as alluded to back in session 03 when we did eigencats.